In [ ]:
from kafka import KafkaProducer
import time

producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: str(v).encode('utf-8')  # simple string encoding
)

for i in range(10):
    message = f"hello world {i}"
    producer.send('example_topic', value=message)
    print(f"Sent: {message}")
    time.sleep(1)

producer.flush()
producer.close()


## using confluent kafka 


In [ ]:
from confluent_kafka import SerializingProducer
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroSerializer
from confluent_kafka.serialization import StringSerializer, SerializationContext, MessageField
import time

# === Configuration ===

conf = {
    'bootstrap.servers': 'your-cluster.kafka.confluent.cloud:9092',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': 'your_kafka_api_key',
    'sasl.password': 'your_kafka_api_secret',
    'key.serializer': StringSerializer('utf_8'),
    'value.serializer': None  # We'll set Avro serializer below
}

schema_registry_conf = {
    'url': 'https://your-schema-registry-url',
    'basic.auth.user.info': 'your_schema_registry_api_key:your_schema_registry_api_secret'
}

# === Schema Registry Client ===
schema_registry_client = SchemaRegistryClient(schema_registry_conf)

# === Avro Schema ===
avro_schema_str = """
{
  "type": "record",
  "name": "User",
  "fields": [
    {"name": "id", "type": "int"},
    {"name": "name", "type": "string"}
  ]
}
"""

# === Avro Serializer ===
avro_serializer = AvroSerializer(
    schema_registry_client=schema_registry_client,
    schema_str=avro_schema_str,
    to_dict=lambda obj, ctx: obj  # No conversion needed if you already pass dict
)

conf['value.serializer'] = avro_serializer

# === Create Producer ===
producer = SerializingProducer(conf)

topic = "example_topic"

for i in range(10):
    value = {"id": i, "name": f"user_{i}"}
    try:
        producer.produce(
            topic=topic,
            key=str(i),
            value=value,
            on_delivery=lambda err, msg: print(
                f"Delivered: {msg.value()} to {msg.topic()} [{msg.partition()}]" if err is None else f"Delivery error: {err}"
            ),
            context=SerializationContext(topic, MessageField.VALUE)
        )
        producer.poll(0)
    except Exception as e:
        print(f"Failed to send: {e}")
    time.sleep(1)

producer.flush()
